# backward-fn-signature — worked example 3: Write subtract_back0 and subtract_back1 with broadcast-reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-fn-signature`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For the binary op `out = x - y` the partials are `d(out)/dx = +1` and `d(out)/dy = -1`. So raw `grad_x = grad_out` and raw `grad_y = -grad_out`. When `x` and `y` broadcast against each other, each returned gradient must be reduced (summed over broadcast axes) back to its input's own shape — the gradient of an input always matches that input's shape.

## Worked solution

**Step 1 — partials of subtraction.** `out = x - y`. Then `d(out)/dx = 1` and `d(out)/dy = -1` elementwise.

**Step 2 — chain rule (raw, pre-reduce).** `grad_x_raw = grad_out * 1 = grad_out`; `grad_y_raw = grad_out * (-1) = -grad_out`. Both have the shape of `out` (the broadcasted shape).

**Step 3 — why we must unbroadcast.** If `x` had shape `(3,)` and `y` had shape `(4, 3)`, then `out` is `(4, 3)` and so is `grad_out`. But `grad_x` must be `(3,)` — the original input shape. Summing over the broadcast (leading / size-1) axes is the gradient of the implicit broadcast/copy operation.

**Step 4 — apply the helper.** `_unbroadcast(grad, target_shape)` first sums away extra leading dims, then sums (with keepdim) any axis where the target was size 1. Apply it to `grad_x_raw` with `x.shape` and to `grad_y_raw` with `y.shape`.

**Step 5 — signature.** Both fns use the extended binary signature `(grad_out, out, x, y)` and return tensors matching `x.shape` and `y.shape` respectively.

In [ ]:
def _unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(dim=0)
    for i, s in enumerate(target_shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def subtract_back0(grad_out, out, x, y):
    # d(x - y)/dx = +1
    return _unbroadcast(grad_out, x.shape)

def subtract_back1(grad_out, out, x, y):
    # d(x - y)/dy = -1
    return _unbroadcast(-grad_out, y.shape)

t.manual_seed(0)
x = t.rand(3)
y = t.rand(4, 3)
out = x - y
grad_out = t.ones_like(out)
gx = subtract_back0(grad_out, out, x, y)
gy = subtract_back1(grad_out, out, x, y)
print('grad_x shape', tuple(gx.shape), 'grad_y shape', tuple(gy.shape))
print('grad_x', gx)          # each entry summed over the 4 broadcast rows
print('grad_y all -1:', t.allclose(gy, -t.ones_like(y)))